# Understanding Pandas 

This notebook is used to practise Pandas library.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
# Confirm that the notebook uses Task 03's virtual environment.
print("Python interpreter:")
print(sys.executable)

print()
# Display the folder from which notebook file paths will be resolved.
print("Current working folder:")
print(Path.cwd())

print()
# Confirm that Pandas is available in this kernel.
print("Pandas version:")
print(pd.__version__)

Python interpreter:
d:\HUMAAA\GitHub\security-automation-learning\03-pandas-credential-stuffing-detector\.venv\Scripts\python.exe

Current working folder:
d:\HUMAAA\GitHub\security-automation-learning\03-pandas-credential-stuffing-detector\notebooks

Pandas version:
3.0.5


In [2]:
from pathlib import Path
import pandas as pd
# Move from the notebooks folder to the Task 03 project folder.
project_root = Path.cwd().parent
# Build the path to the synthetic authentication dataset.
log_file = project_root / "data" / "auth_attempts.csv"
# Load the CSV into a Pandas DataFrame.
logs = pd.read_csv(log_file)
# Convert timestamp text into datetime values.
logs["timestamp"] = pd.to_datetime(
    logs["timestamp"],
    errors="raise",
)

In [3]:
print(logs.head())

            timestamp username      source_ip   result
0 2026-08-02 10:00:00    alice     192.0.2.10  success
1 2026-08-02 10:03:00    alice     192.0.2.10  failure
2 2026-08-02 10:06:00      bob  198.51.100.20  success
3 2026-08-02 10:10:00  charlie   203.0.113.50  failure
4 2026-08-02 10:10:40    diana   203.0.113.50  failure


In [8]:
logs.columns

Index(['timestamp', 'username', 'source_ip', 'result'], dtype='str')

In [ ]:
list(logs.iloc[9])

[Timestamp('2026-08-02 10:20:00'), 'irene', '198.51.100.30', 'failure']

In [16]:
logs.iloc[5]["timestamp"]

Timestamp('2026-08-02 10:11:20')

In [17]:
logs[logs["source_ip"] == "203.0.113.50"]

,timestamp,username,source_ip,result
3,2026-08-02 10:10:00,charlie,203.0.113.50,failure
4,2026-08-02 10:10:40,diana,203.0.113.50,failure
5,2026-08-02 10:11:20,elena,203.0.113.50,failure
6,2026-08-02 10:12:00,farah,203.0.113.50,failure
7,2026-08-02 10:12:40,george,203.0.113.50,failure
8,2026-08-02 10:13:20,harry,203.0.113.50,success


In [18]:
logs[(logs["source_ip"] == "203.0.113.50") & (logs["result"] == "failure")]

,timestamp,username,source_ip,result
3,2026-08-02 10:10:00,charlie,203.0.113.50,failure
4,2026-08-02 10:10:40,diana,203.0.113.50,failure
5,2026-08-02 10:11:20,elena,203.0.113.50,failure
6,2026-08-02 10:12:00,farah,203.0.113.50,failure
7,2026-08-02 10:12:40,george,203.0.113.50,failure


In [ ]:
def count_attempts_by_ip(logs: pd.DataFrame) -> pd.Series:
    # Group rows that have the same source_ip value.
    grouped_ips = logs.groupby("source_ip")
    # Count how many rows belong to each IP address.
    attempt_counts = grouped_ips.size()
    # Sort the results so the busiest IP appears first.
    attempt_counts = attempt_counts.sort_values(ascending=False)

    return attempt_counts

In [32]:
grouped_ips = logs.groupby("source_ip")
attempt_counts = grouped_ips.size()
print(attempt_counts)

source_ip
192.0.2.10       2
198.51.100.20    1
198.51.100.30    1
203.0.113.50     6
dtype: int64


In [33]:
logs.loc[logs["source_ip"] == "203.0.113.50"]

,timestamp,username,source_ip,result
3,2026-08-02 10:10:00,charlie,203.0.113.50,failure
4,2026-08-02 10:10:40,diana,203.0.113.50,failure
5,2026-08-02 10:11:20,elena,203.0.113.50,failure
6,2026-08-02 10:12:00,farah,203.0.113.50,failure
7,2026-08-02 10:12:40,george,203.0.113.50,failure
8,2026-08-02 10:13:20,harry,203.0.113.50,success


In [ ]:
def calculate_failure_metrics(logs: pd.DataFrame) -> pd.DataFrame:
    # Create a new Boolean column, true means the login result was "failure" and false means it wasnt.
    logs = logs.copy()
    logs["is_failure"] = logs["result"].eq("failure")
    # Group the authentication records by source IP.
    grouped_logs = logs.groupby("source_ip")
    # Build a summary table for each IP address.
    failure_metrics = grouped_logs.agg(
        # Count every authentication attempt in the group.
        total_attempts=("result", "size"),
        # Count True values in is_failure.
        # In Python, True behaves like 1 and False behaves like 0.
        failed_attempts=("is_failure", "sum"),
    )
    # Calculate the percentage of attempts that failed.
    failure_metrics["failure_rate"] = (
        failure_metrics["failed_attempts"]
        / failure_metrics["total_attempts"]
    )
    # Sort the IPs so the highest failure rate appears first.
    failure_metrics = failure_metrics.sort_values(
        by="failure_rate",
        ascending=False,
    )

In [26]:
failure_metrics = calculate_failure_metrics(logs)
print(failure_metrics)

               total_attempts  failed_attempts  failure_rate
source_ip                                                   
198.51.100.30               1                1      1.000000
203.0.113.50                6                5      0.833333
192.0.2.10                  2                1      0.500000
198.51.100.20               1                0      0.000000


In [ ]:
def build_five_minute_summary(
    logs: pd.DataFrame,
    window_minutes: int = 5,
) -> pd.DataFrame:
    windowed_logs = logs.copy()
    # Create a Boolean column identifying failed login attempts.
    windowed_logs["is_failure"] = (
        windowed_logs["result"].eq("failure")
    )
    # Pandas frequency string such as "5min".
    time_rule = f"{window_minutes}min"
    # First group the records by source IP.
    # Then resample each IP's records into fixed five-minute time windows.
    # Finally, calculate several measurements for every IP and time window.
    summary = (
        windowed_logs
        .groupby("source_ip")
        .resample(
            time_rule,
            on="timestamp",
        )
        .agg(
            # Count all authentication attempts in the window.
            total_attempts=("result", "size"),
            # Count different usernames attempted in the window.
            unique_users=("username", "nunique"),
            # Count failed login attempts in the window.
            failed_attempts=("is_failure", "sum"),
        )
        .reset_index()
    )
    # Rename the timestamp generated by resample so its meaning is clear.
    summary = summary.rename(
        columns={"timestamp": "window_start"}
    )
    # Calculate the end of each time window.
    summary["window_end"] = (
        summary["window_start"]
        + pd.Timedelta(minutes=window_minutes)
    )
    # Calculate the proportion of attempts that failed.
    summary["failure_rate"] = (
        summary["failed_attempts"]
        / summary["total_attempts"]
    )
    # Arrange the columns into a clear report order.
    summary = summary[
        [
            "source_ip",
            "window_start",
            "window_end",
            "total_attempts",
            "unique_users",
            "failed_attempts",
            "failure_rate",
        ]
    ]
    # Sort the report by IP address and time.
    summary = summary.sort_values(
        by=["source_ip", "window_start"],
    )

    return summary

In [27]:
build5min = build_five_minute_summary(logs)
print(build5min)

       source_ip        window_start          window_end  total_attempts  \
0     192.0.2.10 2026-08-02 10:00:00 2026-08-02 10:05:00               2   
1  198.51.100.20 2026-08-02 10:05:00 2026-08-02 10:10:00               1   
2  198.51.100.30 2026-08-02 10:20:00 2026-08-02 10:25:00               1   
3   203.0.113.50 2026-08-02 10:10:00 2026-08-02 10:15:00               6   

   unique_users  failed_attempts  failure_rate  
0             1                1      0.500000  
1             1                0      0.000000  
2             1                1      1.000000  
3             6                5      0.833333  
